# 🩸 献血ライフマネージャー (HaemaTrack) for Google Colab
**あなたのGoogleサービス上で動く、献血記録の管理・統計分析・可視化ダッシュボード**

---
### 💡 使い方
1. 上部メニューの **「ランタイム」 >「すべてのセルを実行」** を押すだけで、全データ（257件）の分析ダッシュボードが表示されます。
2. Google Drive と連携して自動保存したい場合は、**ステップ 1-B** で Drive をマウントしてください。
3. 新しい献血記録を追加したい場合は、一番下の **「新規献血記録の追加フォーム」** から簡単に追加・保存できます。



## 🛠️ ステップ 1: 環境セットアップ & ライブラリ読み込み


In [ ]:
# 必要ライブラリのインポート
import os
import io
import re
import json
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

print("✅ ライブラリの準備が完了しました！")



## 📂 ステップ 2: 献血データの読み込み
Google Drive内のファイル、アップロードしたファイル、または内蔵の257件のデータを自動読み込みします。



In [ ]:
# 内蔵のデフォルトXMLデータ (257件の履歴データ)
EMBEDDED_XML = '''<?xml version="1.0" ?>
<bloodDonationData version="1.0" exportedAt="2026-07-28T08:04:45">
  <meta>
    <title>献血記録データ</title>
    <author>献血管理アプリ</author>
    <totalCount>257</totalCount>
  </meta>
  <records>
    <record id="1">
      <date>2007-03-09</date>
      <year>2007</year>
      <yearMonth>2007/3</yearMonth>
      <type>全血</type>
      <place>明石</place>
    </record>
    <record id="2">
      <date>2007-09-29</date>
      <year>2007</year>
      <yearMonth>2007/9</yearMonth>
      <type>全血</type>
      <place>ミント</place>
    </record>
    <record id="3">
      <date>2007-12-24</date>
      <year>2007</year>
      <yearMonth>2007/12</yearMonth>
      <type>血漿</type>
      <place>ミント</place>
    </record>
    <record id="4">
      <date>2008-01-27</date>
      <year>2008</year>
      <yearMonth>2008/1</yearMonth>
      <type>血漿</type>
      <place>ミント</place>
    </record>
    <record id="5">
      <date>2008-03-20</date>
      <year>2008</year>
      <yearMonth>2008/3</yearMonth>
      <type>血小板</type>
      <place>ミント</place>
    </record>
    <record id="6">
      <date>2008-04-12</date>
      <year>2008</year>
      <yearMonth>2008/4</yearMonth>
      <type>血小板</type>
      <place>ミント</place>
    </record>
    <record id="7">
      <date>2008-04-27</date>
      <year>2008</year>
      <yearMonth>2008/4</yearMonth>
      <type>血小板</type>
      <place>ミント</place>
    </record>
    <record id="8">
      <date>2008-05-24</date>
      <year>2008</year>
      <yearMonth>2008/5</yearMonth>
      <type>血漿</type>
      <place>ミント</place>
    </record>
    <record id="9">
      <date>2008-06-29</date>
      <year>2008</year>
      <yearMonth>2008/6</yearMonth>
      <type>血漿</type>
      <place>ミント</place>
    </record>
    <record id="10">
      <date>2008-08-02</date>
      <year>2008</year>
      <yearMonth>2008/8</yearMonth>
      <type>血小板</type>
      <place>ミント</place>
    </record>
    <record id="11">
      <date>2008-10-05</date>
      <year>2008</year>
      <yearMonth>2008/10</yearMonth>
      <type>血漿</type>
      <place>ミント</place>
    </record>
    <record id="12">
      <date>2008-11-01</date>
      <year>2008</year>
      <yearMonth>2008/11</yearMonth>
      <type>血漿</type>
      <place>ミント</place>
    </record>
    <record id="13">
      <date>2008-11-16</date>
      <year>2008</year>
      <yearMonth>2008/11</yearMonth>
      <type>血小板</type>
      <place>ミント</place>
    </record>
    <record id="14">
      <date>2009-01-04</date>
      <year>2009</year>
      <yearMonth>2009/1</yearMonth>
      <type>血小板</type>
      <place>ミント</place>
    </record>
    <record id="15">
      <date>2009-02-22</date>
      <year>2009</year>
      <yearMonth>2009/2</yearMonth>
      <type>血小板</type>
      <place>ミント</place>
    </record>
    <record id="16">
      <date>2009-03-20</date>
      <year>2009</year>
      <yearMonth>2009/3</yearMonth>
      <type>血小板</type>
      <place>ミント</place>
    </record>
    <record id="17">
      <date>2009-05-01</date>
      <year>2009</year>
      <yearMonth>2009/5</yearMonth>
      <type>血漿</type>
      <place>ミント</place>
    </record>
    <record id="18">
      <date>2009-06-06</date>
      <year>2009</year>
      <yearMonth>2009/6</yearMonth>
      <type>血漿</type>
      <place>ミント</place>
    </record>
    <record id="19">
      <date>2009-06-20</date>
      <year>2009</year>
      <yearMonth>2009/6</yearMonth>
      <type>血小板</type>
      <place>ミント</place>
    </record>
    <record id="20">
      <date>2009-07-04</date>
      <year>2009</year>
      <yearMonth>2009/7</yearMonth>
      <type>血小板</type>
      <place>ミント</place>
    </record>
    <record id="21">
      <date>2009-07-22</date>
      <year>2009</year>
      <yearMonth>2009/7</yearMonth>
      <type>血小板</type>
      <place>ミント</place>
    </record>
    <record id="22">
      <date>2009-08-15</date>
      <year>2009</year>
      <yearMonth>2009/8</yearMonth>
      <type>血小板</type>
      <place>ミント</place>
    </record>
    <record id="23">
      <date>2009-11-26</date>
      <year>2009</year>
      <yearMonth>2009/11</yearMonth>
      <type>血小板</type>
      <place>ミント</place>
    </record>
    <record id="24">
      <date>2009-12-12</date>
      <year>2009</year>
      <yearMonth>2009/12</yearMonth>
      <type>血小板</type>
      <place>ミント</place>
    </record>
    <record id="25">
      <date>2010-02-25</date>
      <year>2010</year>
      <yearMonth>2010/2</yearMonth>
      <type>血漿</type>
      <place>ミント</place>
    </record>
    <record id="26">
      <date>2010-04-17</date>
      <year>2010</year>
      <yearMonth>2010/4</yearMonth>
      <type>血小板</type>
      <place>ミント</place>
    </record>
    <record id="27">
      <date>2010-05-02</date>
      <year>2010</year>
      <yearMonth>2010/5</yearMonth>
      <type>血漿</type>
      <place>ミント</place>
    </record>
    <record id="28">
      <date>2010-05-22</date>
      <year>2010</year>
      <yearMonth>2010/5</yearMonth>
      <type>血漿</type>
      <place>ミント</place>
    </record>
    <record id="29">
      <date>2010-06-19</date>
      <year>2010</year>
      <yearMonth>2010/6</yearMonth>
      <type>血漿</type>
      <place>ミント</place>
    </record>
    <record id="30">
      <date>2010-07-03</date>
      <year>2010</year>
      <yearMonth>2010/7</yearMonth>
      <type>血小板</type>
      <place>ミント</place>
    </record>
    <record id="31">
      <date>2010-07-19</date>
      <year>2010</year>
      <yearMonth>2010/7</yearMonth>
      <type>血漿</type>
      <place>ミント</place>
    </record>
    <record id="32">
      <date>2010-08-07</date>
      <year>2010</year>
      <yearMonth>2010/8</yearMonth>
      <type>血小板</type>
      <place>ミント</place>
    </record>
    <record id="33">
      <date>2010-08-22</date>
      <year>2010</year>
      <yearMonth>2010/8</yearMonth>
      <type>血漿</type>
      <place>ミント</place>
    </record>
    <record id="34">
      <date>2010-09-23</date>
      <year>2010</year>
      <yearMonth>2010/9</yearMonth>
      <type>血漿</type>
      <place>ミント</place>
    </record>
    <record id="35">
      <date>2010-10-10</date>
      <year>2010</year>
      <yearMonth>2010/10</yearMonth>
      <type>血小板</type>
      <place>ミント</place>
    </record>
    <record id="36">
      <date>2010-11-13</date>
      <year>2010</year>
      <yearMonth>2010/11</yearMonth>
      <type>血小板</type>
      <place>ミント</place>
    </record>
    <record id="37">
      <date>2010-11-27</date>
      <year>2010</year>
      <yearMonth>2010/11</yearMonth>
      <type>血小板</type>
      <place>ミント</place>
    </record>
    <record id="38">
      <date>2011-03-13</date>
      <year>2011</year>
      <yearMonth>2011/3</yearMonth>
      <type>血漿</type>
      <place>ミント</place>
    </record>
    <record id="39">
      <date>2011-05-05</date>
      <year>2011</year>
      <yearMonth>2011/5</yearMonth>
      <type>血漿</type>
      <place>ミント</place>
    </record>
    <record id="40">
      <date>2011-06-26</date>
      <year>2011</year>
      <yearMonth>2011/6</yearMonth>
      <type>血漿</type>
      <place>ミント</place>
    </record>
    <record id="41">
      <date>2011-07-16</date>
      <year>2011</year>
      <yearMonth>2011/7</yearMonth>
      <type>血漿</type>
      <place>ミント</place>
    </record>
    <record id="42">
      <date>2011-08-21</date>
      <year>2011</year>
      <yearMonth>2011/8</yearMonth>
      <type>血漿</type>
      <place>ミント</place>
    </record>
    <record id="43">
      <date>2011-09-17</date>
      <year>2011</year>
      <yearMonth>2011/9</yearMonth>
      <type>血漿</type>
      <place>ミント</place>
    </record>
    <record id="44">
      <date>2011-10-09</date>
      <year>2011</year>
      <yearMonth>2011/10</yearMonth>
      <type>血漿</type>
      <place>ミント</place>
    </record>
    <record id="45">
      <date>2011-11-03</date>
      <year>2011</year>
      <yearMonth>2011/11</yearMonth>
      <type>血小板</type>
      <place>ミント</place>
    </record>
    <record id="46">
      <date>2011-11-20</date>
      <year>2011</year>
      <yearMonth>2011/11</yearMonth>
      <type>血小板</type>
      <place>ミント</place>
    </record>
    <record id="47">
      <date>2011-12-17</date>
      <year>2011</year>
      <yearMonth>2011/12</yearMonth>
      <type>血小板</type>
      <place>ミント</place>
    </record>
    <record id="48">
      <date>2012-01-08</date>
      <year>2012</year>
      <yearMonth>2012/1</yearMonth>
      <type>血小板</type>
      <place>ミント</place>
    </record>
    <record id="49">
      <date>2012-01-28</date>
      <year>2012</year>
      <yearMonth>2012/1</yearMonth>
      <type>血漿</type>
      <place>ミント</place>
    </record>
    <record id="50">
      <date>2012-02-18</date>
      <year>2012</year>
      <yearMonth>2012/2</yearMonth>
      <type>血漿</type>
      <place>大阪府</place>
    </record>
    <record id="51">
      <date>2012-03-20</date>
      <year>2012</year>
      <yearMonth>2012/3</yearMonth>
      <type>血小板</type>
      <place>ミント</place>
    </record>
    <record id="52">
      <date>2012-04-22</date>
      <year>2012</year>
      <yearMonth>2012/4</yearMonth>
      <type>血小板</type>
      <place>阪急グランドビル</place>
    </record>
    <record id="53">
      <date>2012-05-26</date>
      <year>2012</year>
      <yearMonth>2012/5</yearMonth>
      <type>血漿</type>
      <place>ミント</place>
    </record>
    <record id="54">
      <date>2012-08-04</date>
      <year>2012</year>
      <yearMonth>2012/8</yearMonth>
      <type>血漿</type>
      <place>ミント</place>
    </record>
    <record id="55">
      <date>2012-09-01</date>
      <year>2012</year>
      <yearMonth>2012/9</yearMonth>
      <type>血小板</type>
      <place>ミント</place>
    </record>
    <record id="56">
      <date>2012-10-08</date>
      <year>2012</year>
      <yearMonth>2012/10</yearMonth>
      <type>血小板</type>
      <place>ミント</place>
    </record>
    <record id="57">
      <date>2012-11-24</date>
      <year>2012</year>
      <yearMonth>2012/11</yearMonth>
      <type>血小板</type>
      <place>ミント</place>
    </record>
    <record id="58">
      <date>2012-12-19</date>
      <year>2012</year>
      <yearMonth>2012/12</yearMonth>
      <type>血小板</type>
      <place>ミント</place>
    </record>
    <record id="59">
      <date>2013-01-14</date>
      <year>2013</year>
      <yearMonth>2013/1</yearMonth>
      <type>血小板</type>
      <place>さんさん</place>
    </record>
    <record id="60">
      <date>2013-01-31</date>
      <year>2013</year>
      <yearMonth>2013/1</yearMonth>
      <type>血小板</type>
      <place>ミント</place>
    </record>
    <record id="61">
      <date>2013-03-10</date>
      <year>2013</year>
      <yearMonth>2013/3</yearMonth>
      <type>血小板</type>
      <place>ミント</place>
    </record>
    <record id="62">
      <date>2013-04-06</date>
      <year>2013</year>
      <yearMonth>2013/4</yearMonth>
      <type>血小板</type>
      <place>にしきた</place>
    </record>
    <record id="63">
      <date>2013-04-21</date>
      <year>2013</year>
      <yearMonth>2013/4</yearMonth>
      <type>血小板</type>
      <place>さんさん</place>
    </record>
    <record id="64">
      <date>2013-05-05</date>
      <year>2013</year>
      <yearMonth>2013/5</yearMonth>
      <type>血小板</type>
      <place>さんさん</place>
    </record>
    <record id="65">
      <date>2013-05-19</date>
      <year>2013</year>
      <yearMonth>2013/5</yearMonth>
      <type>血小板</type>
      <place>にしきた</place>
    </record>
    <record id="66">
      <date>2013-06-02</date>
      <year>2013</year>
      <yearMonth>2013/6</yearMonth>
      <type>血漿</type>
      <place>にしきた</place>
    </record>
    <record id="67">
      <date>2013-08-10</date>
      <year>2013</year>
      <yearMonth>2013/8</yearMonth>
      <type>血漿</type>
      <place>さんさん</place>
    </record>
    <record id="68">
      <date>2013-09-01</date>
      <year>2013</year>
      <yearMonth>2013/9</yearMonth>
      <type>血小板</type>
      <place>にしきた</place>
    </record>
    <record id="69">
      <date>2013-10-12</date>
      <year>2013</year>
      <yearMonth>2013/10</yearMonth>
      <type>血小板</type>
      <place>ミント</place>
    </record>
    <record id="70">
      <date>2013-11-24</date>
      <year>2013</year>
      <yearMonth>2013/11</yearMonth>
      <type>血小板</type>
      <place>にしきた</place>
    </record>
    <record id="71">
      <date>2013-12-21</date>
      <year>2013</year>
      <yearMonth>2013/12</yearMonth>
      <type>血小板</type>
      <place>にしきた</place>
    </record>
    <record id="72">
      <date>2014-01-19</date>
      <year>2014</year>
      <yearMonth>2014/1</yearMonth>
      <type>血小板</type>
      <place>にしきた</place>
    </record>
    <record id="73">
      <date>2014-02-02</date>
      <year>2014</year>
      <yearMonth>2014/2</yearMonth>
      <type>血小板</type>
      <place>にしきた</place>
    </record>
    <record id="74">
      <date>2014-03-11</date>
      <year>2014</year>
      <yearMonth>2014/3</yearMonth>
      <type>血小板</type>
      <place>ミント</place>
    </record>
    <record id="75">
      <date>2014-04-06</date>
      <year>2014</year>
      <yearMonth>2014/4</yearMonth>
      <type>血小板</type>
      <place>にしきた</place>
    </record>
    <record id="76">
      <date>2014-04-27</date>
      <year>2014</year>
      <yearMonth>2014/4</yearMonth>
      <type>血小板</type>
      <place>にしきた</place>
    </record>
    <record id="77">
      <date>2014-06-22</date>
      <year>2014</year>
      <yearMonth>2014/6</yearMonth>
      <type>血小板</type>
      <place>ミント</place>
    </record>
    <record id="78">
      <date>2014-07-06</date>
      <year>2014</year>
      <yearMonth>2014/7</yearMonth>
      <type>血小板</type>
      <place>にしきた</place>
    </record>
    <record id="79">
      <date>2014-07-20</date>
      <year>2014</year>
      <yearMonth>2014/7</yearMonth>
      <type>血漿</type>
      <place>にしきた</place>
    </record>
    <record id="80">
      <date>2014-08-10</date>
      <year>2014</year>
      <yearMonth>2014/8</yearMonth>
      <type>血漿</type>
      <place>にしきた</place>
    </record>
    <record id="81">
      <date>2014-09-06</date>
      <year>2014</year>
      <yearMonth>2014/9</yearMonth>
      <type>血小板</type>
      <place>ミント</place>
    </record>
    <record id="82">
      <date>2014-10-12</date>
      <year>2014</year>
      <yearMonth>2014/10</yearMonth>
      <type>血小板</type>
      <place>ミント</place>
    </record>
    <record id="83">
      <date>2014-11-30</date>
      <year>2014</year>
      <yearMonth>2014/11</yearMonth>
      <type>血小板</type>
      <place>にしきた</place>
    </record>
    <record id="84">
      <date>2014-12-21</date>
      <year>2014</year>
      <yearMonth>2014/12</yearMonth>
      <type>血小板</type>
      <place>ミント</place>
    </record>
    <record id="85">
      <date>2015-01-24</date>
      <year>2015</year>
      <yearMonth>2015/1</yearMonth>
      <type>血漿</type>
      <place>にしきた</place>
    </record>
    <record id="86">
      <date>2015-02-07</date>
      <year>2015</year>
      <yearMonth>2015/2</yearMonth>
      <type>血小板</type>
      <place>さんさん</place>
    </record>
    <record id="87">
      <date>2015-02-22</date>
      <year>2015</year>
      <yearMonth>2015/2</yearMonth>
      <type>血漿</type>
      <place>にしきた</place>
    </record>
    <record id="88">
      <date>2015-03-15</date>
      <year>2015</year>
      <yearMonth>2015/3</yearMonth>
      <type>血小板</type>
      <place>さんさん</place>
    </record>
    <record id="89">
      <date>2015-04-12</date>
      <year>2015</year>
      <yearMonth>2015/4</yearMonth>
      <type>血漿</type>
      <place>さんさん</place>
    </record>
    <record id="90">
      <date>2015-04-26</date>
      <year>2015</year>
      <yearMonth>2015/4</yearMonth>
      <type>血漿</type>
      <place>さんさん</place>
    </record>
    <record id="91">
      <date>2015-05-10</date>
      <year>2015</year>
      <yearMonth>2015/5</yearMonth>
      <type>血小板</type>
      <place>にしきた</place>
    </record>
    <record id="92">
      <date>2015-06-27</date>
      <year>2015</year>
      <yearMonth>2015/6</yearMonth>
      <type>血小板</type>
      <place>にしきた</place>
    </record>
    <record id="93">
      <date>2015-07-12</date>
      <year>2015</year>
      <yearMonth>2015/7</yearMonth>
      <type>血漿</type>
      <place>ミント</place>
    </record>
    <record id="94">
      <date>2015-07-26</date>
      <year>2015</year>
      <yearMonth>2015/7</yearMonth>
      <type>血小板</type>
      <place>さんさん</place>
    </record>
    <record id="95">
      <date>2015-08-13</date>
      <year>2015</year>
      <yearMonth>2015/8</yearMonth>
      <type>血漿</type>
      <place>さんさん</place>
    </record>
    <record id="96">
      <date>2015-09-12</date>
      <year>2015</year>
      <yearMonth>2015/9</yearMonth>
      <type>血小板</type>
      <place>にしきた</place>
    </record>
    <record id="97">
      <date>2015-10-19</date>
      <year>2015</year>
      <yearMonth>2015/10</yearMonth>
      <type>血小板</type>
      <place>にしきた</place>
    </record>
    <record id="98">
      <date>2015-12-01</date>
      <year>2015</year>
      <yearMonth>2015/12</yearMonth>
      <type>血小板</type>
      <place>にしきた</place>
    </record>
    <record id="99">
      <date>2015-12-22</date>
      <year>2015</year>
      <yearMonth>2015/12</yearMonth>
      <type>血小板</type>
      <place>にしきた</place>
    </record>
    <record id="100">
      <date>2016-01-25</date>
      <year>2016</year>
      <yearMonth>2016/1</yearMonth>
      <type>血漿</type>
      <place>にしきた</place>
    </record>
    <record id="101">
      <date>2016-02-11</date>
      <year>2016</year>
      <yearMonth>2016/2</yearMonth>
      <type>血小板</type>
      <place>にしきた</place>
    </record>
    <record id="102">
      <date>2016-02-28</date>
      <year>2016</year>
      <yearMonth>2016/2</yearMonth>
      <type>血漿</type>
      <place>にしきた</place>
    </record>
    <record id="103">
      <date>2016-03-15</date>
      <year>2016</year>
      <yearMonth>2016/3</yearMonth>
      <type>血小板</type>
      <place>にしきた</place>
    </record>
    <record id="104">
      <date>2016-04-16</date>
      <year>2016</year>
      <yearMonth>2016/4</yearMonth>
      <type>血漿</type>
      <place>にしきた</place>
    </record>
    <record id="105">
      <date>2016-05-01</date>
      <year>2016</year>
      <yearMonth>2016/5</yearMonth>
      <type>血漿</type>
      <place>さんさん</place>
    </record>
    <record id="106">
      <date>2016-05-15</date>
      <year>2016</year>
      <yearMonth>2016/5</yearMonth>
      <type>血小板</type>
      <place>にしきた</place>
    </record>
    <record id="107">
      <date>2016-07-03</date>
      <year>2016</year>
      <yearMonth>2016/7</yearMonth>
      <type>血小板</type>
      <place>さんさん</place>
    </record>
    <record id="108">
      <date>2016-07-17</date>
      <year>2016</year>
      <yearMonth>2016/7</yearMonth>
      <type>血漿</type>
      <place>にしきた</place>
    </record>
    <record id="109">
      <date>2016-08-07</date>
      <year>2016</year>
      <yearMonth>2016/8</yearMonth>
      <type>血小板</type>
      <place>にしきた</place>
    </record>
    <record id="110">
      <date>2016-08-21</date>
      <year>2016</year>
      <yearMonth>2016/8</yearMonth>
      <type>血漿</type>
      <place>にしきた</place>
    </record>
    <record id="111">
      <date>2016-09-18</date>
      <year>2016</year>
      <yearMonth>2016/9</yearMonth>
      <type>血小板</type>
      <place>ミント</place>
    </record>
    <record id="112">
      <date>2016-10-22</date>
      <year>2016</year>
      <yearMonth>2016/10</yearMonth>
      <type>血小板</type>
      <place>にしきた</place>
    </record>
    <record id="113">
      <date>2016-12-04</date>
      <year>2016</year>
      <yearMonth>2016/12</yearMonth>
      <type>血小板</type>
      <place>にしきた</place>
    </record>
    <record id="114">
      <date>2016-12-23</date>
      <year>2016</year>
      <yearMonth>2016/12</yearMonth>
      <type>血漿</type>
      <place>さんさん</place>
    </record>
    <record id="115">
      <date>2017-01-08</date>
      <year>2017</year>
      <yearMonth>2017/1</yearMonth>
      <type>血漿</type>
      <place>さんさん</place>
    </record>
    <record id="116">
      <date>2017-01-28</date>
      <year>2017</year>
      <yearMonth>2017/1</yearMonth>
      <type>血漿</type>
      <place>にしきた</place>
    </record>
    <record id="117">
      <date>2017-02-11</date>
      <year>2017</year>
      <yearMonth>2017/2</yearMonth>
      <type>血漿</type>
      <place>にしきた</place>
    </record>
    <record id="118">
      <date>2017-02-25</date>
      <year>2017</year>
      <yearMonth>2017/2</yearMonth>
      <type>血漿</type>
      <place>にしきた</place>
    </record>
    <record id="119">
      <date>2017-03-11</date>
      <year>2017</year>
      <yearMonth>2017/3</yearMonth>
      <type>血漿</type>
      <place>にしきた</place>
    </record>
    <record id="120">
      <date>2017-03-25</date>
      <year>2017</year>
      <yearMonth>2017/3</yearMonth>
      <type>血漿</type>
      <place>にしきた</place>
    </record>
    <record id="121">
      <date>2017-04-08</date>
      <year>2017</year>
      <yearMonth>2017/4</yearMonth>
      <type>血漿</type>
      <place>ミント</place>
    </record>
    <record id="122">
      <date>2017-04-22</date>
      <year>2017</year>
      <yearMonth>2017/4</yearMonth>
      <type>血漿</type>
      <place>にしきた</place>
    </record>
    <record id="123">
      <date>2017-05-06</date>
      <year>2017</year>
      <yearMonth>2017/5</yearMonth>
      <type>血漿</type>
      <place>にしきた</place>
    </record>
    <record id="124">
      <date>2017-05-20</date>
      <year>2017</year>
      <yearMonth>2017/5</yearMonth>
      <type>血漿</type>
      <place>にしきた</place>
    </record>
    <record id="125">
      <date>2017-06-04</date>
      <year>2017</year>
      <yearMonth>2017/6</yearMonth>
      <type>血漿</type>
      <place>にしきた</place>
    </record>
    <record id="126">
      <date>2017-07-06</date>
      <year>2017</year>
      <yearMonth>2017/7</yearMonth>
      <type>血漿</type>
      <place>にしきた</place>
    </record>
    <record id="127">
      <date>2017-07-23</date>
      <year>2017</year>
      <yearMonth>2017/7</yearMonth>
      <type>血小板</type>
      <place>にしきた</place>
    </record>
    <record id="128">
      <date>2017-08-19</date>
      <year>2017</year>
      <yearMonth>2017/8</yearMonth>
      <type>血小板</type>
      <place>にしきた</place>
    </record>
    <record id="129">
      <date>2017-09-09</date>
      <year>2017</year>
      <yearMonth>2017/9</yearMonth>
      <type>血漿</type>
      <place>にしきた</place>
    </record>
    <record id="130">
      <date>2017-10-08</date>
      <year>2017</year>
      <yearMonth>2017/10</yearMonth>
      <type>血漿</type>
      <place>にしきた</place>
    </record>
    <record id="131">
      <date>2018-01-07</date>
      <year>2018</year>
      <yearMonth>2018/1</yearMonth>
      <type>血漿</type>
      <place>にしきた</place>
    </record>
    <record id="132">
      <date>2018-01-21</date>
      <year>2018</year>
      <yearMonth>2018/1</yearMonth>
      <type>血小板</type>
      <place>にしきた</place>
    </record>
    <record id="133">
      <date>2018-02-18</date>
      <year>2018</year>
      <yearMonth>2018/2</yearMonth>
      <type>血小板</type>
      <place>ミント</place>
    </record>
    <record id="134">
      <date>2018-03-31</date>
      <year>2018</year>
      <yearMonth>2018/3</yearMonth>
      <type>血小板</type>
      <place>にしきた</place>
    </record>
    <record id="135">
      <date>2018-04-30</date>
      <year>2018</year>
      <yearMonth>2018/4</yearMonth>
      <type>血小板</type>
      <place>にしきた</place>
    </record>
    <record id="136">
      <date>2020-07-09</date>
      <year>2020</year>
      <yearMonth>2020/7</yearMonth>
      <type>全血</type>
      <place>明石</place>
    </record>
    <record id="137">
      <date>2020-10-04</date>
      <year>2020</year>
      <yearMonth>2020/10</yearMonth>
      <type>全血</type>
      <place>センタープラザ</place>
    </record>
    <record id="138">
      <date>2020-12-29</date>
      <year>2020</year>
      <yearMonth>2020/12</yearMonth>
      <type>全血</type>
      <place>にしきた</place>
    </record>
    <record id="139">
      <date>2021-02-28</date>
      <year>2021</year>
      <yearMonth>2021/2</yearMonth>
      <type>血漿</type>
      <place>ミント</place>
    </record>
    <record id="140">
      <date>2021-03-14</date>
      <year>2021</year>
      <yearMonth>2021/3</yearMonth>
      <type>血小板</type>
      <place>さんさん</place>
    </record>
    <record id="141">
      <date>2021-03-28</date>
      <year>2021</year>
      <yearMonth>2021/3</yearMonth>
      <type>血漿</type>
      <place>にしきた</place>
    </record>
    <record id="142">
      <date>2021-04-18</date>
      <year>2021</year>
      <yearMonth>2021/4</yearMonth>
      <type>血漿</type>
      <place>ミント</place>
    </record>
    <record id="143">
      <date>2021-05-16</date>
      <year>2021</year>
      <yearMonth>2021/5</yearMonth>
      <type>血小板</type>
      <place>さんさん</place>
    </record>
    <record id="144">
      <date>2021-06-07</date>
      <year>2021</year>
      <yearMonth>2021/6</yearMonth>
      <type>血小板</type>
      <place>にしきた</place>
    </record>
    <record id="145">
      <date>2021-06-26</date>
      <year>2021</year>
      <yearMonth>2021/6</yearMonth>
      <type>血小板</type>
      <place>にしきた</place>
    </record>
    <record id="146">
      <date>2021-07-10</date>
      <year>2021</year>
      <yearMonth>2021/7</yearMonth>
      <type>全血</type>
      <place>にしきた</place>
    </record>
    <record id="147">
      <date>2021-09-05</date>
      <year>2021</year>
      <yearMonth>2021/9</yearMonth>
      <type>血小板</type>
      <place>さんさん</place>
    </record>
    <record id="148">
      <date>2021-09-20</date>
      <year>2021</year>
      <yearMonth>2021/9</yearMonth>
      <type>血小板</type>
      <place>ミント</place>
    </record>
    <record id="149">
      <date>2021-10-09</date>
      <year>2021</year>
      <yearMonth>2021/10</yearMonth>
      <type>血小板</type>
      <place>にしきた</place>
    </record>
    <record id="150">
      <date>2021-10-23</date>
      <year>2021</year>
      <yearMonth>2021/10</yearMonth>
      <type>血漿</type>
      <place>にしきた</place>
    </record>
    <record id="151">
      <date>2021-11-07</date>
      <year>2021</year>
      <yearMonth>2021/11</yearMonth>
      <type>血小板</type>
      <place>さんさん</place>
    </record>
    <record id="152">
      <date>2021-11-21</date>
      <year>2021</year>
      <yearMonth>2021/11</yearMonth>
      <type>血漿</type>
      <place>ミント</place>
    </record>
    <record id="153">
      <date>2021-12-05</date>
      <year>2021</year>
      <yearMonth>2021/12</yearMonth>
      <type>血小板</type>
      <place>にしきた</place>
    </record>
    <record id="154">
      <date>2021-12-19</date>
      <year>2021</year>
      <yearMonth>2021/12</yearMonth>
      <type>血漿</type>
      <place>にしきた</place>
    </record>
    <record id="155">
      <date>2022-01-02</date>
      <year>2022</year>
      <yearMonth>2022/1</yearMonth>
      <type>全血</type>
      <place>にしきた</place>
    </record>
    <record id="156">
      <date>2022-02-27</date>
      <year>2022</year>
      <yearMonth>2022/2</yearMonth>
      <type>血漿</type>
      <place>にしきた</place>
    </record>
    <record id="157">
      <date>2022-03-13</date>
      <year>2022</year>
      <yearMonth>2022/3</yearMonth>
      <type>血漿</type>
      <place>にしきた</place>
    </record>
    <record id="158">
      <date>2022-03-27</date>
      <year>2022</year>
      <yearMonth>2022/3</yearMonth>
      <type>血小板</type>
      <place>にしきた</place>
    </record>
    <record id="159">
      <date>2022-04-17</date>
      <year>2022</year>
      <yearMonth>2022/4</yearMonth>
      <type>血漿</type>
      <place>三重</place>
    </record>
    <record id="160">
      <date>2022-05-15</date>
      <year>2022</year>
      <yearMonth>2022/5</yearMonth>
      <type>血小板</type>
      <place>三重</place>
    </record>
    <record id="161">
      <date>2022-06-12</date>
      <year>2022</year>
      <yearMonth>2022/6</yearMonth>
      <type>血小板</type>
      <place>三重</place>
    </record>
    <record id="162">
      <date>2022-06-26</date>
      <year>2022</year>
      <yearMonth>2022/6</yearMonth>
      <type>血漿</type>
      <place>三重</place>
    </record>
    <record id="163">
      <date>2022-07-10</date>
      <year>2022</year>
      <yearMonth>2022/7</yearMonth>
      <type>血漿</type>
      <place>三重</place>
    </record>
    <record id="164">
      <date>2022-07-24</date>
      <year>2022</year>
      <yearMonth>2022/7</yearMonth>
      <type>全血</type>
      <place>三重</place>
    </record>
    <record id="165">
      <date>2022-09-18</date>
      <year>2022</year>
      <yearMonth>2022/9</yearMonth>
      <type>血小板</type>
      <place>三重</place>
    </record>
    <record id="166">
      <date>2022-10-02</date>
      <year>2022</year>
      <yearMonth>2022/10</yearMonth>
      <type>血漿</type>
      <place>三重</place>
    </record>
    <record id="167">
      <date>2022-10-16</date>
      <year>2022</year>
      <yearMonth>2022/10</yearMonth>
      <type>血漿</type>
      <place>三重</place>
    </record>
    <record id="168">
      <date>2022-10-30</date>
      <year>2022</year>
      <yearMonth>2022/10</yearMonth>
      <type>血漿</type>
      <place>四日市</place>
    </record>
    <record id="169">
      <date>2022-11-13</date>
      <year>2022</year>
      <yearMonth>2022/11</yearMonth>
      <type>血漿</type>
      <place>三重</place>
    </record>
    <record id="170">
      <date>2022-11-27</date>
      <year>2022</year>
      <yearMonth>2022/11</yearMonth>
      <type>血小板</type>
      <place>三重</place>
    </record>
    <record id="171">
      <date>2022-12-11</date>
      <year>2022</year>
      <yearMonth>2022/12</yearMonth>
      <type>血小板</type>
      <place>三重</place>
    </record>
    <record id="172">
      <date>2022-12-25</date>
      <year>2022</year>
      <yearMonth>2022/12</yearMonth>
      <type>血漿</type>
      <place>三重</place>
    </record>
    <record id="173">
      <date>2023-01-08</date>
      <year>2023</year>
      <yearMonth>2023/1</yearMonth>
      <type>血漿</type>
      <place>三重</place>
    </record>
    <record id="174">
      <date>2023-01-22</date>
      <year>2023</year>
      <yearMonth>2023/1</yearMonth>
      <type>血漿</type>
      <place>三重</place>
    </record>
    <record id="175">
      <date>2023-02-26</date>
      <year>2023</year>
      <yearMonth>2023/2</yearMonth>
      <type>血漿</type>
      <place>三重</place>
    </record>
    <record id="176">
      <date>2023-03-12</date>
      <year>2023</year>
      <yearMonth>2023/3</yearMonth>
      <type>血漿</type>
      <place>四日市</place>
    </record>
    <record id="177">
      <date>2023-03-26</date>
      <year>2023</year>
      <yearMonth>2023/3</yearMonth>
      <type>血漿</type>
      <place>三重</place>
    </record>
    <record id="178">
      <date>2023-04-09</date>
      <year>2023</year>
      <yearMonth>2023/4</yearMonth>
      <type>血漿</type>
      <place>三重</place>
    </record>
    <record id="179">
      <date>2023-04-23</date>
      <year>2023</year>
      <yearMonth>2023/4</yearMonth>
      <type>血漿</type>
      <place>三重</place>
    </record>
    <record id="180">
      <date>2023-05-14</date>
      <year>2023</year>
      <yearMonth>2023/5</yearMonth>
      <type>血漿</type>
      <place>三重</place>
    </record>
    <record id="181">
      <date>2023-05-28</date>
      <year>2023</year>
      <yearMonth>2023/5</yearMonth>
      <type>血漿</type>
      <place>伊勢</place>
    </record>
    <record id="182">
      <date>2023-06-11</date>
      <year>2023</year>
      <yearMonth>2023/6</yearMonth>
      <type>血漿</type>
      <place>三重</place>
    </record>
    <record id="183">
      <date>2023-06-25</date>
      <year>2023</year>
      <yearMonth>2023/6</yearMonth>
      <type>血漿</type>
      <place>三重</place>
    </record>
    <record id="184">
      <date>2023-07-09</date>
      <year>2023</year>
      <yearMonth>2023/7</yearMonth>
      <type>血漿</type>
      <place>三重</place>
    </record>
    <record id="185">
      <date>2023-07-23</date>
      <year>2023</year>
      <yearMonth>2023/7</yearMonth>
      <type>血漿</type>
      <place>三重</place>
    </record>
    <record id="186">
      <date>2023-09-17</date>
      <year>2023</year>
      <yearMonth>2023/9</yearMonth>
      <type>血漿</type>
      <place>三重</place>
    </record>
    <record id="187">
      <date>2023-10-01</date>
      <year>2023</year>
      <yearMonth>2023/10</yearMonth>
      <type>血漿</type>
      <place>三重</place>
    </record>
    <record id="188">
      <date>2023-10-15</date>
      <year>2023</year>
      <yearMonth>2023/10</yearMonth>
      <type>血漿</type>
      <place>三重</place>
    </record>
    <record id="189">
      <date>2023-10-29</date>
      <year>2023</year>
      <yearMonth>2023/10</yearMonth>
      <type>血漿</type>
      <place>伊勢</place>
    </record>
    <record id="190">
      <date>2023-11-12</date>
      <year>2023</year>
      <yearMonth>2023/11</yearMonth>
      <type>血漿</type>
      <place>三重</place>
    </record>
    <record id="191">
      <date>2023-11-26</date>
      <year>2023</year>
      <yearMonth>2023/11</yearMonth>
      <type>血漿</type>
      <place>三重</place>
    </record>
    <record id="192">
      <date>2023-12-10</date>
      <year>2023</year>
      <yearMonth>2023/12</yearMonth>
      <type>血漿</type>
      <place>三重</place>
    </record>
    <record id="193">
      <date>2023-12-24</date>
      <year>2023</year>
      <yearMonth>2023/12</yearMonth>
      <type>血漿</type>
      <place>三重</place>
    </record>
    <record id="194">
      <date>2024-01-07</date>
      <year>2024</year>
      <yearMonth>2024/1</yearMonth>
      <type>血漿</type>
      <place>三重</place>
    </record>
    <record id="195">
      <date>2024-01-21</date>
      <year>2024</year>
      <yearMonth>2024/1</yearMonth>
      <type>血漿</type>
      <place>三重</place>
    </record>
    <record id="196">
      <date>2024-02-04</date>
      <year>2024</year>
      <yearMonth>2024/2</yearMonth>
      <type>血漿</type>
      <place>伊勢</place>
    </record>
    <record id="197">
      <date>2024-02-18</date>
      <year>2024</year>
      <yearMonth>2024/2</yearMonth>
      <type>血漿</type>
      <place>三重</place>
    </record>
    <record id="198">
      <date>2024-03-03</date>
      <year>2024</year>
      <yearMonth>2024/3</yearMonth>
      <type>血漿</type>
      <place>三重</place>
    </record>
    <record id="199">
      <date>2024-03-17</date>
      <year>2024</year>
      <yearMonth>2024/3</yearMonth>
      <type>血漿</type>
      <place>三重</place>
    </record>
    <record id="200">
      <date>2024-03-31</date>
      <year>2024</year>
      <yearMonth>2024/3</yearMonth>
      <type>血漿</type>
      <place>伊勢</place>
    </record>
    <record id="201">
      <date>2024-04-14</date>
      <year>2024</year>
      <yearMonth>2024/4</yearMonth>
      <type>血漿</type>
      <place>伊勢</place>
    </record>
    <record id="202">
      <date>2024-04-28</date>
      <year>2024</year>
      <yearMonth>2024/4</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
    <record id="203">
      <date>2024-05-12</date>
      <year>2024</year>
      <yearMonth>2024/5</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
    <record id="204">
      <date>2024-05-26</date>
      <year>2024</year>
      <yearMonth>2024/5</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
    <record id="205">
      <date>2024-06-09</date>
      <year>2024</year>
      <yearMonth>2024/6</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
    <record id="206">
      <date>2024-06-23</date>
      <year>2024</year>
      <yearMonth>2024/6</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
    <record id="207">
      <date>2024-07-07</date>
      <year>2024</year>
      <yearMonth>2024/7</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
    <record id="208">
      <date>2024-07-21</date>
      <year>2024</year>
      <yearMonth>2024/7</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
    <record id="209">
      <date>2024-08-04</date>
      <year>2024</year>
      <yearMonth>2024/8</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
    <record id="210">
      <date>2024-09-15</date>
      <year>2024</year>
      <yearMonth>2024/9</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
    <record id="211">
      <date>2024-09-29</date>
      <year>2024</year>
      <yearMonth>2024/9</yearMonth>
      <type>血漿</type>
      <place>四日市</place>
    </record>
    <record id="212">
      <date>2024-10-13</date>
      <year>2024</year>
      <yearMonth>2024/10</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
    <record id="213">
      <date>2024-10-27</date>
      <year>2024</year>
      <yearMonth>2024/10</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
    <record id="214">
      <date>2024-11-10</date>
      <year>2024</year>
      <yearMonth>2024/11</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
    <record id="215">
      <date>2024-11-24</date>
      <year>2024</year>
      <yearMonth>2024/11</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
    <record id="216">
      <date>2024-12-08</date>
      <year>2024</year>
      <yearMonth>2024/12</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
    <record id="217">
      <date>2024-12-22</date>
      <year>2024</year>
      <yearMonth>2024/12</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
    <record id="218">
      <date>2025-01-05</date>
      <year>2025</year>
      <yearMonth>2025/1</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
    <record id="219">
      <date>2025-01-19</date>
      <year>2025</year>
      <yearMonth>2025/1</yearMonth>
      <type>血漿</type>
      <place>伊勢</place>
    </record>
    <record id="220">
      <date>2025-02-02</date>
      <year>2025</year>
      <yearMonth>2025/2</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
    <record id="221">
      <date>2025-02-16</date>
      <year>2025</year>
      <yearMonth>2025/2</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
    <record id="222">
      <date>2025-03-02</date>
      <year>2025</year>
      <yearMonth>2025/3</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
    <record id="223">
      <date>2025-03-16</date>
      <year>2025</year>
      <yearMonth>2025/3</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
    <record id="224">
      <date>2025-03-30</date>
      <year>2025</year>
      <yearMonth>2025/3</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
    <record id="225">
      <date>2025-04-13</date>
      <year>2025</year>
      <yearMonth>2025/4</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
    <record id="226">
      <date>2025-04-27</date>
      <year>2025</year>
      <yearMonth>2025/4</yearMonth>
      <type>血漿</type>
      <place>福井</place>
    </record>
    <record id="227">
      <date>2025-05-11</date>
      <year>2025</year>
      <yearMonth>2025/5</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
    <record id="228">
      <date>2025-05-25</date>
      <year>2025</year>
      <yearMonth>2025/5</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
    <record id="229">
      <date>2025-06-08</date>
      <year>2025</year>
      <yearMonth>2025/6</yearMonth>
      <type>血漿</type>
      <place>伊勢</place>
    </record>
    <record id="230">
      <date>2025-06-22</date>
      <year>2025</year>
      <yearMonth>2025/6</yearMonth>
      <type>血漿</type>
      <place>伊勢</place>
    </record>
    <record id="231">
      <date>2025-07-06</date>
      <year>2025</year>
      <yearMonth>2025/7</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
    <record id="232">
      <date>2025-07-20</date>
      <year>2025</year>
      <yearMonth>2025/7</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
    <record id="233">
      <date>2025-08-03</date>
      <year>2025</year>
      <yearMonth>2025/8</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
    <record id="234">
      <date>2025-09-14</date>
      <year>2025</year>
      <yearMonth>2025/9</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
    <record id="235">
      <date>2025-09-28</date>
      <year>2025</year>
      <yearMonth>2025/9</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
    <record id="236">
      <date>2025-10-12</date>
      <year>2025</year>
      <yearMonth>2025/10</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
    <record id="237">
      <date>2025-10-26</date>
      <year>2025</year>
      <yearMonth>2025/10</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
    <record id="238">
      <date>2025-11-09</date>
      <year>2025</year>
      <yearMonth>2025/11</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
    <record id="239">
      <date>2025-11-23</date>
      <year>2025</year>
      <yearMonth>2025/11</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
    <record id="240">
      <date>2025-12-17</date>
      <year>2025</year>
      <yearMonth>2025/12</yearMonth>
      <type>血漿</type>
      <place>伊勢</place>
    </record>
    <record id="241">
      <date>2025-12-21</date>
      <year>2025</year>
      <yearMonth>2025/12</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
    <record id="242">
      <date>2026-01-04</date>
      <year>2026</year>
      <yearMonth>2026/1</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
    <record id="243">
      <date>2026-01-23</date>
      <year>2026</year>
      <yearMonth>2026/1</yearMonth>
      <type>血漿</type>
      <place>姫路</place>
    </record>
    <record id="244">
      <date>2026-02-07</date>
      <year>2026</year>
      <yearMonth>2026/2</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
    <record id="245">
      <date>2026-02-22</date>
      <year>2026</year>
      <yearMonth>2026/2</yearMonth>
      <type>血漿</type>
      <place>伊勢</place>
    </record>
    <record id="246">
      <date>2026-03-08</date>
      <year>2026</year>
      <yearMonth>2026/3</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
    <record id="247">
      <date>2026-03-28</date>
      <year>2026</year>
      <yearMonth>2026/3</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
    <record id="248">
      <date>2026-04-11</date>
      <year>2026</year>
      <yearMonth>2026/4</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
    <record id="249">
      <date>2026-04-25</date>
      <year>2026</year>
      <yearMonth>2026/4</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
    <record id="250">
      <date>2026-05-09</date>
      <year>2026</year>
      <yearMonth>2026/5</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
    <record id="251">
      <date>2026-05-23</date>
      <year>2026</year>
      <yearMonth>2026/5</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
    <record id="252">
      <date>2026-06-06</date>
      <year>2026</year>
      <yearMonth>2026/6</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
    <record id="253">
      <date>2026-06-27</date>
      <year>2026</year>
      <yearMonth>2026/6</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
    <record id="254">
      <date>2026-07-11</date>
      <year>2026</year>
      <yearMonth>2026/7</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
    <record id="255">
      <date>2026-07-25</date>
      <year>2026</year>
      <yearMonth>2026/7</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
    <record id="256">
      <date>2026-08-08</date>
      <year>2026</year>
      <yearMonth>2026/8</yearMonth>
      <type>血漿</type>
      <place>三重（新）</place>
    </record>
      <record id="257">
      <date>2026-08-22</date>
      <year>2026</year>
      <yearMonth>2026/8</yearMonth>
      <type>血漿</type>
      <place>伊勢</place>
    </record>
  </records>
</bloodDonationData>
'''

def parse_xml_records(xml_text):
    """XML文字列からPandas DataFrameを生成"""
    root = ET.fromstring(xml_text)
    records = []
    for rec in root.findall('.//record'):
        rec_id = int(rec.get('id', len(records) + 1))
        date_str = rec.findtext('date', '').strip()
        year_str = rec.findtext('year', '').strip()
        year_month = rec.findtext('yearMonth', '').strip()
        blood_type = rec.findtext('type', '').strip()
        place = rec.findtext('place', '').strip()
        memo = rec.findtext('memo', '').strip()
        
        if date_str:
            try:
                date_dt = pd.to_datetime(date_str)
                year_val = int(year_str) if year_str.isdigit() else date_dt.year
                records.append({
                    'id': rec_id,
                    'date': date_dt,
                    'date_str': date_str,
                    'year': year_val,
                    'year_month': year_month if year_month else f"{date_dt.year}/{date_dt.month}",
                    'type': blood_type,
                    'place': place if place else '未指定',
                    'memo': memo
                })
            except Exception as e:
                pass
                
    df = pd.DataFrame(records)
    if not df.empty:
        df = df.sort_values('date').reset_index(drop=True)
    return df

# データの自動ロード処理
DATA_FILE_PATH = "kenketsu_data.xml"

if os.path.exists(DATA_FILE_PATH):
    print(f"📁 ローカルファイル '{DATA_FILE_PATH}' から読み込みます...")
    with open(DATA_FILE_PATH, 'r', encoding='utf-8') as f:
        xml_content = f.read()
else:
    print("✨ 内蔵の献血データ（257件）をロードします...")
    xml_content = EMBEDDED_XML
    # 作業ディレクトリに保存しておく
    with open(DATA_FILE_PATH, 'w', encoding='utf-8') as f:
        f.write(xml_content)

df_records = parse_xml_records(xml_content)
print(f"🎉 読み込み完了: 合計 {len(df_records)} 件の献血データがセットされました！")



### ☁️ (オプション) Google Driveと連携して保存する場合
Google Driveに `kenketsu_data.xml` を永続保存したい場合は、以下のセルを実行してDriveをマウントしてください。



In [ ]:
# Google Drive マウント (必要な場合のみ実行)
try:
    from google.colab import drive
    # drive.mount('/content/drive')
    print("💡 Google Driveをマウントする場合は、上の行のコメント `#` を外して実行してください。")
except ImportError:
    print("※ ローカルJupyter環境で実行中です。")



## 📊 ステップ 3: 献血サマリー & 次回献血可能日ダッシュボード


In [ ]:
def calculate_kpis(df):
    if df.empty:
        return {}
    
    total_count = len(df)
    first_row = df.iloc[0]
    latest_row = df.iloc[-1]
    
    first_date = first_row['date']
    latest_date = latest_row['date']
    today = pd.to_datetime('today').normalize()
    
    # 献血歴 (年・月・日)
    total_days = (today - first_date).days
    years_elapsed = total_days // 365
    months_elapsed = (total_days % 365) // 30
    
    # 概算献血量 (400mL全血:400, 200mL:200, 成分献血:一律400mL換算基準)
    def calc_volume(row):
        t = str(row['type'])
        if '200' in t: return 200
        elif '全血' in t or '400' in t: return 400
        elif '血小板' in t or '血漿' in t: return 400 # 成分献血
        return 400
    
    total_volume_ml = df.apply(calc_volume, axis=1).sum()
    total_volume_l = total_volume_ml / 1000.0
    
    # 年間平均ペース
    active_year_span = max(1, df['year'].max() - df['year'].min() + 1)
    yearly_avg = total_count / active_year_span
    
    # 次回献血可能日の計算 (赤十字ルール)
    # 成分献血(血漿・血小板): 2週間後(14日) / 全血400mL(男性12週・女性16週、一般目安8〜12週)
    latest_type = latest_row['type']
    if '全血' in latest_type:
        wait_days = 56 # 8週間 (次回成分または全血可能基準)
    else:
        wait_days = 14 # 2週間 (次回成分献血可能基準)
        
    next_avail_date = latest_date + timedelta(days=wait_days)
    days_until_next = (next_avail_date - today).days
    can_donate_now = days_until_next <= 0
    
    # 献血間隔の平均
    df_sorted = df.sort_values('date')
    intervals = (df_sorted['date'] - df_sorted['date'].shift(1)).dt.days.dropna()
    avg_interval = intervals.mean() if not intervals.empty else 0
    
    return {
        'total_count': total_count,
        'total_volume_l': total_volume_l,
        'first_date': first_date.strftime('%Y年%m月%d日'),
        'latest_date': latest_date.strftime('%Y年%m月%d日'),
        'latest_place': latest_row['place'],
        'latest_type': latest_type,
        'years_elapsed': years_elapsed,
        'months_elapsed': months_elapsed,
        'yearly_avg': yearly_avg,
        'next_avail_date': next_avail_date.strftime('%Y年%m月%d日'),
        'days_until_next': days_until_next,
        'can_donate_now': can_donate_now,
        'avg_interval': avg_interval
    }

kpi = calculate_kpis(df_records)

# リッチなHTMLカードでダッシュボード表示
html_content = f"""
<div style="font-family: 'Segoe UI', Meiryo, sans-serif; background: #0f172a; padding: 24px; border-radius: 16px; color: #f8fafc; margin-bottom: 20px;">
  <div style="display: flex; justify-content: space-between; align-items: center; border-bottom: 1px solid #334155; padding-bottom: 16px; margin-bottom: 20px;">
    <h2 style="margin: 0; color: #f43f5e; font-size: 1.6rem; display: flex; align-items: center; gap: 8px;">
      🩸 献血実績サマリーダッシュボード
    </h2>
    <span style="background: #1e293b; padding: 6px 14px; border-radius: 20px; font-size: 0.85rem; color: #94a3b8; border: 1px solid #475569;">
      最新記録: {kpi['latest_date']} ({kpi['latest_place']})
    </span>
  </div>

  <div style="display: grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap: 16px; margin-bottom: 24px;">
    <!-- 通算回数 -->
    <div style="background: #1e293b; padding: 18px; border-radius: 12px; border-left: 4px solid #f43f5e;">
      <div style="color: #94a3b8; font-size: 0.85rem; font-weight: bold;">通算献血回数</div>
      <div style="font-size: 2.2rem; font-weight: 800; color: #f8fafc; margin: 4px 0;">{kpi['total_count']} <span style="font-size: 1rem; font-weight: normal; color: #cbd5e1;">回</span></div>
      <div style="color: #38bdf8; font-size: 0.8rem;">🎉 年平均 {kpi['yearly_avg']:.1f} 回ペース</div>
    </div>

    <!-- 累計献血量 -->
    <div style="background: #1e293b; padding: 18px; border-radius: 12px; border-left: 4px solid #38bdf8;">
      <div style="color: #94a3b8; font-size: 0.85rem; font-weight: bold;">累計献血量（推計）</div>
      <div style="font-size: 2.2rem; font-weight: 800; color: #f8fafc; margin: 4px 0;">{kpi['total_volume_l']:.1f} <span style="font-size: 1rem; font-weight: normal; color: #cbd5e1;">L</span></div>
      <div style="color: #a855f7; font-size: 0.8rem;">2Lペットボトル約 {kpi['total_volume_l']/2:.1f} 本分</div>
    </div>

    <!-- 献血歴 -->
    <div style="background: #1e293b; padding: 18px; border-radius: 12px; border-left: 4px solid #10b981;">
      <div style="color: #94a3b8; font-size: 0.85rem; font-weight: bold;">献血歴 (初献血日)</div>
      <div style="font-size: 1.5rem; font-weight: 700; color: #f8fafc; margin: 6px 0;">{kpi['years_elapsed']}年 {kpi['months_elapsed']}ヶ月</div>
      <div style="color: #94a3b8; font-size: 0.8rem;">初: {kpi['first_date']}</div>
    </div>

    <!-- 次回可能日 -->
    <div style="background: {'#064e3b' if kpi['can_donate_now'] else '#1e293b'}; padding: 18px; border-radius: 12px; border-left: 4px solid {'#10b981' if kpi['can_donate_now'] else '#eab308'};">
      <div style="color: #94a3b8; font-size: 0.85rem; font-weight: bold;">次回献血可能日</div>
      <div style="font-size: 1.3rem; font-weight: 700; color: {'#34d399' if kpi['can_donate_now'] else '#fde047'}; margin: 6px 0;">
        {'✅ 今すぐ献血可能です！' if kpi['can_donate_now'] else kpi['next_avail_date']}
      </div>
      <div style="color: #cbd5e1; font-size: 0.8rem;">
        {'（献血可能状態です）' if kpi['can_donate_now'] else f"あと {kpi['days_until_next']} 日で次回可能"}
      </div>
    </div>
  </div>
</div>
"""
display(HTML(html_content))



## 📈 ステップ 4: インタラクティブ・ビジュアル分析グラフ

マウスをホバーすると詳細な数値が表示され、拡大・縮小や凡例のクリックで絞り込みができます。



In [ ]:
# 1. 年別推移 & 献血種別 (Stacked Bar + Trend)
yearly_type_df = df_records.groupby(['year', 'type']).size().reset_index(name='count')

fig_yearly = px.bar(
    yearly_type_df,
    x='year',
    y='count',
    color='type',
    title='📅 年別 献血回数推移（種別別積み上げ）',
    labels={'year': '年', 'count': '献血回数 (回)', 'type': '献血種別'},
    color_discrete_map={'全血': '#ef4444', '血小板': '#3b82f6', '血漿': '#10b981', 'その他': '#8b5cf6'},
    text='count'
)
fig_yearly.update_layout(
    barmode='stack',
    template='plotly_dark',
    xaxis=dict(type='category', title='年'),
    yaxis=dict(title='回数 (回)'),
    hovermode='x unified',
    height=450
)
fig_yearly.show()

# 2. 献血種別シェア (Donut Chart) & 献血場所ランキング Top 10
fig_pie_bar = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "domain"}, {"type": "xy"}]],
    subplot_titles=("🩸 献血種別の構成比", "🏢 献血ルーム・場所ランキング Top 10")
)

# 種別比率
type_counts = df_records['type'].value_counts().reset_index()
type_counts.columns = ['type', 'count']
fig_pie_bar.add_trace(
    go.Pie(
        labels=type_counts['type'],
        values=type_counts['count'],
        hole=0.45,
        marker=dict(colors=['#3b82f6', '#10b981', '#ef4444', '#8b5cf6']),
        textinfo='label+percent+value'
    ),
    row=1, col=1
)

# 場所ランキング
place_counts = df_records['place'].value_counts().head(10).reset_index()
place_counts.columns = ['place', 'count']
place_counts = place_counts.sort_values('count', ascending=True)

fig_pie_bar.add_trace(
    go.Bar(
        x=place_counts['count'],
        y=place_counts['place'],
        orientation='h',
        marker=dict(color='#f43f5e'),
        text=place_counts['count'],
        textposition='auto'
    ),
    row=1, col=2
)

fig_pie_bar.update_layout(
    template='plotly_dark',
    height=420,
    showlegend=False
)
fig_pie_bar.show()

# 3. 月別 & 曜日別 献血傾向ヒートマップ
df_records['month'] = df_records['date'].dt.month
df_records['weekday_name'] = df_records['date'].dt.day_name()
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekday_ja = ['月', '火', '水', '木', '金', '土', '日']
df_records['weekday_ja'] = df_records['date'].dt.dayofweek.map(lambda x: weekday_ja[x])

month_counts = df_records['month'].value_counts().sort_index().reset_index()
month_counts.columns = ['month', 'count']
month_counts['month_label'] = month_counts['month'].astype(str) + '月'

weekday_counts = df_records['weekday_ja'].value_counts().reindex(weekday_ja).fillna(0).reset_index()
weekday_counts.columns = ['weekday', 'count']

fig_season = make_subplots(
    rows=1, cols=2,
    subplot_titles=("🗓️ 月別 献血頻度", "📆 曜日別 献血頻度")
)

fig_season.add_trace(
    go.Bar(
        x=month_counts['month_label'],
        y=month_counts['count'],
        marker=dict(color='#06b6d4'),
        text=month_counts['count'],
        textposition='auto'
    ),
    row=1, col=1
)

fig_season.add_trace(
    go.Bar(
        x=weekday_counts['weekday'],
        y=weekday_counts['count'],
        marker=dict(color='#a855f7'),
        text=weekday_counts['count'],
        textposition='auto'
    ),
    row=1, col=2
)

fig_season.update_layout(
    template='plotly_dark',
    height=380,
    showlegend=False
)
fig_season.show()

# 4. 累積献血回数の成長曲線
df_sorted = df_records.sort_values('date').copy()
df_sorted['cumulative_count'] = range(1, len(df_sorted) + 1)

fig_growth = px.area(
    df_sorted,
    x='date',
    y='cumulative_count',
    title='🚀 通算献血回数の成長軌跡（タイムライン）',
    labels={'date': '日付', 'cumulative_count': '通算回数 (回)'},
    template='plotly_dark'
)
fig_growth.update_traces(line_color='#f43f5e', fillcolor='rgba(244, 63, 94, 0.25)')
fig_growth.update_layout(height=400, hovermode='x unified')
fig_growth.show()



## ✍️ ステップ 5: 新規献血記録の追加フォーム & データ保存
新しい献血をした際は、以下のフォームに入力して **「➕ 新しい記録を追加して保存」** ボタンを押すだけで、自動的にデータが更新され XML / Excel としてダウンロード・保存できます。



In [ ]:
# 新規入力ウィジェット
today_str = datetime.now().strftime('%Y-%m-%d')
date_picker = widgets.DatePicker(
    description='献血日:',
    value=datetime.now().date(),
    disabled=False
)

type_dropdown = widgets.Dropdown(
    options=['血小板', '血漿', '全血', '400mL', '200mL', 'その他'],
    value='血小板',
    description='種別:'
)

# 過去の場所リストを取得
existing_places = sorted(list(df_records['place'].unique()))
place_combobox = widgets.Combobox(
    placeholder='例: ミント、明石、梅田など',
    options=existing_places,
    description='場所:',
    ensure_option=False,
    disabled=False,
    value=existing_places[0] if existing_places else ''
)

memo_text = widgets.Text(
    value='',
    placeholder='メモ（記念品、体調、検査数値など）',
    description='メモ:'
)

save_button = widgets.Button(
    description='➕ 新しい記録を追加して保存',
    button_style='danger',
    tooltip='新規記録を反映してXML/Excelを更新します',
    icon='plus'
)

out = widgets.Output()

def build_xml_string(df):
    """DataFrameからXML文字列を生成"""
    root = ET.Element('bloodDonationData', version='1.0', exportedAt=datetime.now().strftime('%Y-%m-%dT%H:%M:%S'))
    meta = ET.SubElement(root, 'meta')
    ET.SubElement(meta, 'title').text = '献血記録データ'
    ET.SubElement(meta, 'author').text = '献血ライフマネージャー Colab'
    ET.SubElement(meta, 'totalCount').text = str(len(df))
    
    records_node = ET.SubElement(root, 'records')
    for idx, row in df.sort_values('date').iterrows():
        rec = ET.SubElement(records_node, 'record', id=str(idx + 1))
        d_str = row['date'].strftime('%Y-%m-%d') if hasattr(row['date'], 'strftime') else str(row['date'])
        y_str = str(row['year'])
        ym_str = str(row['year_month'])
        
        ET.SubElement(rec, 'date').text = d_str
        ET.SubElement(rec, 'year').text = y_str
        ET.SubElement(rec, 'yearMonth').text = ym_str
        ET.SubElement(rec, 'type').text = str(row['type'])
        ET.SubElement(rec, 'place').text = str(row['place'])
        if row['memo']:
            ET.SubElement(rec, 'memo').text = str(row['memo'])
            
    xml_str = ET.tostring(root, encoding='utf-8')
    import xml.dom.minidom
    dom = xml.dom.minidom.parseString(xml_str)
    return dom.toprettyxml(indent='  ')

def on_save_clicked(b):
    global df_records
    with out:
        clear_output()
        if not date_picker.value:
            print("⚠️ 献血日を選択してください。")
            return
            
        new_date = pd.to_datetime(date_picker.value)
        new_type = type_dropdown.value
        new_place = place_combobox.value.strip() or '未指定'
        new_memo = memo_text.value.strip()
        
        new_id = len(df_records) + 1
        new_row = {
            'id': new_id,
            'date': new_date,
            'date_str': new_date.strftime('%Y-%m-%d'),
            'year': new_date.year,
            'year_month': f"{new_date.year}/{new_date.month}",
            'type': new_type,
            'place': new_place,
            'memo': new_memo
        }
        
        # DataFrame更新
        df_records = pd.concat([df_records, pd.DataFrame([new_row])], ignore_index=True)
        df_records = df_records.sort_values('date').reset_index(drop=True)
        
        # XMLファイル書き出し
        xml_output = build_xml_string(df_records)
        with open('kenketsu_data.xml', 'w', encoding='utf-8') as f:
            f.write(xml_output)
            
        # Excelファイル書き出し
        df_export = df_records[['id', 'date_str', 'year', 'year_month', 'type', 'place', 'memo']].copy()
        df_export.columns = ['通算回数', '献血日', '年', '年月', '種別', '献血場所', '備考']
        df_export.to_excel('kenketsu_records.xlsx', index=False)
        
        print(f"🎉 成功！ 通算第 {len(df_records)} 回の献血記録（{new_date.strftime('%Y-%m-%d')} / {new_type} / {new_place}）を追加保存しました！")
        print("📁 'kenketsu_data.xml' と 'kenketsu_records.xlsx' に反映されました。")
        print("💡 次回可能日やグラフを更新するには、上のセルを再実行してください。")

save_button.on_click(on_save_clicked)

form_box = widgets.VBox([
    widgets.HTML("<h3>📝 新しい献血記録を入力</h3>"),
    date_picker,
    type_dropdown,
    place_combobox,
    memo_text,
    widgets.HBox([save_button]),
    out
])
display(form_box)



## 📋 ステップ 6: 献血記録一覧テーブル & 検索


In [ ]:
# データ一覧のテーブル表示
display_df = df_records[['id', 'date_str', 'type', 'place', 'memo']].copy()
display_df.columns = ['通算No', '献血日', '種別', '献血場所', 'メモ']
display_df = display_df.sort_values('通算No', ascending=False)

print(f"📖 全 {len(display_df)} 件の献血記録一覧（最新順）:")
display(display_df.head(20)) # 直近20件を表示

